# Páramos — Gestión, Monitoreo y Delimitación de Ecosistemas de Páramo

## Qué es esta línea y por qué importa en Colombia

Los páramos son ecosistemas de alta montaña exclusivos del neotrópico. Colombia alberga aproximadamente el **50% de los páramos del mundo** y depende de ellos para regular hasta el **70% del recurso hídrico** del país. No son simples montañas frías: sus suelos orgánicos profundos (andosoles), su vegetación especializada (frailejones, musgos, pajonales) y su densa nubosidad los convierten en las verdaderas "fábricas de agua" que abastecen acueductos, generan energía hidroeléctrica y sostienen la agricultura en tierras bajas. El complejo Sumapaz, por ejemplo, es el páramo más grande del mundo y fuente principal del acueducto de Bogotá.

La amenaza es concreta: minería de carbón y oro, ganadería extensiva y cultivos de papa han avanzado sobre estos ecosistemas durante décadas. La respuesta institucional está consagrada en la **Ley 1930 de 2018 (Ley de Páramos)**, que prohíbe actividades extractivas, ordena planes de saneamiento predial y establece regímenes de sustitución de actividades agropecuarias. La delimitación oficial se realiza a escala **1:25.000** bajo liderazgo del MADS con insumos del Instituto Humboldt (IAvH).

## Preguntas que este notebook responde

- ¿Existe una tendencia estadísticamente significativa de calentamiento en la serie de temperatura del páramo analizado?
- ¿Cuál es la influencia del fenómeno ENSO (El Niño / La Niña) sobre la temperatura con un lag de 2 meses?
- ¿Los modelos SARIMA, Prophet o XGBoost capturan mejor la estacionalidad bimodal del ciclo hidrológico colombiano (abr-may, oct-nov)?

## Instituciones clave

| Institución | Rol principal |
|---|---|
| MADS | Delimitación oficial y normativa de páramos |
| Instituto Humboldt (IAvH) | Atlas de páramos a escala 1:100.000, insumos biológicos |
| IDEAM | Redes hidrometeorológicas, SISAIRE |
| IGAC | Cartografía base, valoración ambiental predial (Res. 963/2025) |
| CARs regionales | Ejecución territorial, planes de manejo y sustitución |

> **Contexto de dominio completo:** [`docs/fuentes/paramos.md`](../../docs/fuentes/paramos.md)  
> **Bloque:** A — Gestión | **Línea:** `paramos`  
> **Variable principal:** `temperatura` (°C) | Rango páramo: **-5 a 16 °C**  
> **Modelos sugeridos:** SARIMA, PROPHET, XGBOOST  
> **ENSO lag:** 2 meses (`config.ENSO_LAG_MESES["paramos"]`)  
> **Métricas primarias:** MAE, RMSE, Mann-Kendall (obligatorio para evidencia de cambio climático)

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "paramos"
VARIABLE = "temperatura"
UNIDAD = "°C"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `paramos` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "paramos.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

Las series de temperatura en páramos provienen principalmente de las redes hidrometeorológicas del **IDEAM** y de las **Corporaciones Autónomas Regionales (CARs)** — en particular CAR Cundinamarca, CORPOGUAVIO y CORPOBOYACÁ para la cordillera Oriental. Las estaciones suelen ser automáticas, con resolución horaria agregada a nivel diario o mensual.

**Rangos físicos válidos para páramos colombianos:**

| Variable | Rango típico | Fuente |
|---|---|---|
| Temperatura media | -5 °C a 16 °C | IDEAM / estaciones de alta montaña |
| Precipitación anual | 500 a >3.000 mm/año | Estaciones pluviométricas IDEAM |
| Cobertura vegetal | 0 a 100 % | Imágenes Sentinel-2 / Landsat |
| Humedad del suelo | 54 a >60 % | Muestreos de campo / laboratorio |

> **Nota de validación (ADR-006):** `validate(df, linea_tematica="paramos")` aplica automáticamente estos rangos y marcará como sospechosos los valores fuera de límites. No descarte outliers sin revisar el contexto meteorológico: una temperatura de 0 °C en enero en el complejo Chingaza puede ser un evento real de helada, no un error de sensor. Ver **ADR-002** sobre política de outliers.

> **Ruta esperada para datos reales:** `data/raw/paramos.csv`  
> Columnas mínimas: `fecha` (datetime), `temperatura` (°C), opcionalmente `precipitacion` (mm), `cobertura_pct` (%).

In [ ]:
# df = load_csv("data/raw/paramos.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "temperatura": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA

La validación de calidad en páramos tiene particularidades críticas que distinguen esta línea de otras:

**Por qué `validate()` con `linea_tematica="paramos"` (ADR-006):** La función aplica rangos físicos específicos del dominio páramo (temperatura entre -5 y 16 °C, precipitación entre 0 y 3.000 mm). Sin esto, un validador genérico podría marcar como error una temperatura de 2 °C que es perfectamente normal en el superpáramo del Cocuy, o aceptar 30 °C que indicarían claramente un fallo de sensor.

**Problemas frecuentes en datos de alta montaña:**
- Datos faltantes por nubosidad persistente en sensores remotos o fallas de comunicación en estaciones automáticas
- Saltos abruptos por cambios en la ubicación o calibración del sensor
- Subestimación de la "precipitación horizontal" (niebla captada por frailejones) que no se mide con pluviómetro convencional

El reporte EDA se guarda en `data/output/reports/eda_paramos.html`. Revisar especialmente: porcentaje de faltantes por mes y distribución estacional de temperatura.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_paramos.html",
       title="EDA — Páramos", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

La visualización de series de temperatura en páramos debe buscar dos señales simultáneamente:

1. **Tendencia de largo plazo:** ¿La temperatura media anual está subiendo? Los estudios del IDEAM y el IPCC reportan que los páramos colombianos se han calentado entre **0.2 y 0.5 °C por década** desde 1970. Esta señal puede ser sutil en 10 años de datos pero se vuelve estadísticamente significativa en series de 30+ años.

2. **Estacionalidad bimodal:** Colombia tiene dos temporadas húmedas (marzo-mayo, septiembre-noviembre) y dos secas (diciembre-febrero, junio-agosto). En los páramos, la temperatura tiende a ser más baja en las temporadas húmedas por mayor nubosidad. La gráfica de medias estacionales (`plot_seasonal_means`) debe reflejar este patrón de doble valle.

Si los datos son de una sola estación, la serie puede contener efectos de microclima local (radiación directa en laderas vs. fondos de cuenca). Comparar con estaciones vecinas del IDEAM cuando sea posible.

In [ ]:
plot_series(df, "fecha", "temperatura", title="Páramos — temperatura (°C)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "temperatura", period="month")
plt.show()

## 3b. Covariable ENSO (ONI) — por qué 2 meses de lag en páramos

El **Oceanic Niño Index (ONI)** mide la anomalía de temperatura superficial del mar en el Pacífico central (región Niño 3.4). Es el índice estándar de NOAA para clasificar eventos El Niño y La Niña. Su relevancia para los páramos colombianos es directa:

- **El Niño** (ONI > +0.5 °C por 5 meses consecutivos): genera **déficit hídrico** en los páramos de las cordilleras Central y Oriental. Las precipitaciones caen entre 20 y 40 % por debajo del promedio histórico. Los caudales base disminuyen dramáticamente, afectando los acueductos que dependen de estas fuentes.
- **La Niña** (ONI < -0.5 °C): genera **exceso hídrico**. Asociado con mayor nubosidad, más baja temperatura y riesgo de deslizamientos en zonas de páramo degradado.

**Por qué lag de 2 meses (ADR-007):** La respuesta hidrológica y térmica de los páramos no es instantánea al forzamiento oceánico. La señal del Pacífico llega transformada por la circulación atmosférica. La literatura colombiana (IDEAM, estudios ENSO-Colombia) identifica un rezago promedio de **2 meses** para la respuesta de temperatura en páramos. Para caudales, el lag es generalmente mayor (3-4 meses).

`enso_lagged()` lee automáticamente `config.ENSO_LAG_MESES["paramos"]` = 2 y agrega las columnas `oni_lag2` y `enso_fase_lag2` al DataFrame. Usar siempre esta función en lugar de construir el lag manualmente (ADR-007).

In [ ]:
# --- Covariable ENSO (lag específico para Páramos) ---
oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica="paramos")
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

En el contexto de páramos, los estadísticos descriptivos tienen interpretaciones concretas:

- **Media y mediana de temperatura:** En páramos andinos la media debería estar entre 7 y 13 °C. Una media más alta puede indicar que la estación está ubicada en transición hacia bosque altoandino (por debajo del límite del páramo) o que hay un sesgo de sensor.
- **Desviación estándar:** La alta variación intradiurna es característica del páramo (hasta 15-20 °C entre la noche y el mediodía en días despejados). Si se trabaja con datos diarios promediados, esto se suaviza, pero en datos horarios es la norma.
- **Coeficiente de variación (CV):** Para precipitación, un CV alto (>80 %) refleja la irregularidad del régimen pluviométrico paramuno. Para temperatura, un CV moderado (20-40 %) es esperable.
- **Percentiles:** Los percentiles 5 y 95 delimitan el "rango operativo" de la estación. Registros por debajo del P5 pueden ser eventos de helada real; por encima del P95, eventos de radiación directa intensa.

No usar estos estadísticos para decidir automáticamente el recorte de outliers. Ver **ADR-002**: en ecosistemas de alta montaña los eventos extremos son señal ecológica, no ruido estadístico.

In [ ]:
summarize(df)

## 5. Análisis inferencial — Estacionariedad y tendencia de cambio climático

Esta sección es la más crítica metodológicamente para la línea de páramos, porque sus resultados tienen implicaciones directas de política pública:

### 5a. Prueba de estacionariedad (ADR-004)

**Por qué ADF + KPSS juntos y no solo ADF (ADR-004):** La prueba ADF (Augmented Dickey-Fuller) tiene baja potencia ante tendencias determinísticas suaves, exactamente el tipo de señal que se espera en temperatura con cambio climático. KPSS complementa: rechaza la hipótesis nula de estacionariedad cuando existe tendencia. El par ADF + KPSS permite distinguir tres escenarios:

| ADF | KPSS | Conclusión |
|---|---|---|
| No rechaza H₀ | Rechaza H₀ | Serie no estacionaria — integrada I(1) o con tendencia |
| Rechaza H₀ | No rechaza H₀ | Serie estacionaria |
| Ambos rechazan | — | Posible raíz estacional — revisar ACF |
| Ninguno rechaza | — | Ambigüedad — series cortas o efecto ENSO |

Para ARIMA, si la serie no es estacionaria, aplicar diferenciación (d=1). Si la temperatura ya tiene tendencia lineal significativa, considerar ARIMA con tendencia o Prophet con `growth="linear"`.

### 5b. Mann-Kendall — detección de tendencia de cambio climático

**Por qué Mann-Kendall es obligatorio en páramos:** A diferencia de las pruebas de regresión paramétrica, Mann-Kendall no asume normalidad de los residuos y es robusto ante valores faltantes y outliers. Es el estándar metodológico del IDEAM y del IPCC para evaluar tendencias en series climáticas. El parámetro `slope` de Sen (output del test) da la **tasa de cambio en °C/mes**, directamente comparable con reportes de calentamiento.

**Interpretación aplicada:**
- `trend = "increasing"` con `p < 0.05`: evidencia estadística de calentamiento — documentar en `docs/decisiones.md` con fecha y slope
- `slope` de Sen expresado en °C/año = slope_mensual × 12
- Una slope de 0.003 °C/mes equivale a **0.36 °C/década** — tasa consistente con reportes IDEAM para páramos

In [ ]:
ts = df.set_index("fecha")["temperatura"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas

Para temperatura en páramos **no existe una norma de emisión colombiana con umbral fijo** como en calidad del aire. Sin embargo, `exceedance_report()` puede usarse de forma personalizada para evaluar cuántas observaciones superan umbrales ecológicos de referencia:

**Umbrales ecológicos de referencia para temperatura en páramos:**

| Umbral | Valor | Significado ecológico |
|---|---|---|
| Límite superior del páramo | 16 °C | Por encima, el ecosistema transita a subpáramo o bosque altoandino |
| Umbral de helada | 0 °C | Por debajo, riesgo para vegetación joven y cultivos en zona de amortiguación |
| Temperatura óptima frailejón | 4 - 12 °C | Rango de desarrollo óptimo de *Espeletia* spp. |

Aunque estos umbrales no son de cumplimiento normativo estricto, monitorear su excedencia frecuencia es parte del **índice de grado de conservación (IGC)** que aplica el IGAC para valoración ambiental predial (Resolución 963 de 2025). Una estación que registra más del 15 % de observaciones por encima de 16 °C está reportando condiciones ajenas al ecosistema de páramo.

Para variables con norma explícita (PM2.5, PM10, ozono, caudales mínimos), usar `exceedance_report(serie, variable="pm25")` con las variables de `config.NORMA_CO` o `config.NORMA_OMS` (ADR-008).

In [ ]:
rep = exceedance_report(df["temperatura"], variable="temperatura")
if rep.empty:
    print("Sin normas colombianas registradas para 'temperatura'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento — Tratamiento de datos faltantes en páramos

La imputación en series ambientales de alta montaña requiere cuidado especial. El método `"linear"` es adecuado para brechas cortas (1-3 meses). Para brechas más largas, considerar:

- **Interpolación estacional:** usar la media del mismo mes en años anteriores (evita que una brecha en época de La Niña sea rellenada con valores de año normal)
- **Imputación por estación vecina:** si existe otra estación IDEAM a menos de 20 km y altitud similar, usar regresión para imputar

**Qué NO hacer (ADR-002):** No aplicar clipping automático sobre los outliers detectados en el paso anterior. Picos de temperatura > 14 °C durante El Niño intenso (2015-2016, por ejemplo) son señal real del calentamiento acelerado en condiciones de sequía. Eliminarlos sesga la tendencia detectada por Mann-Kendall.

El `df_clean` resultante se usa directamente en la sección de modelos. Las columnas ENSO (oni_lag2) deben preservarse como covariables.

In [ ]:
df_clean = impute(df.copy(), cols=["temperatura"], method="linear")
print(f"Faltantes antes: {df["temperatura"].isna().sum()} | "
      f"después: {df_clean["temperatura"].isna().sum()}")

## 7. Modelos predictivos — Selección metodológica para páramos

### Por qué estos tres modelos

**SARIMA** es el punto de partida obligatorio para series temporales de temperatura mensual con estacionalidad anual (período 12). Captura la autocorrelación temporal y la estacionalidad bimodal colombiana. Su limitación es que no modela directamente el efecto ENSO como covariable; para eso usar SARIMAX con `oni_lag2` como regresor exógeno.

**Prophet** (Meta/Facebook) es particularmente útil en páramos porque: (a) maneja datos faltantes sin imputación previa, (b) permite incorporar el componente de tendencia con cambio de punto (`changepoints`) que puede capturar el calentamiento, y (c) su componente de estacionalidad anual es flexible para el régimen bimodal colombiano. Configurar `growth="linear"` si Mann-Kendall detectó tendencia.

**XGBoost** con lags captura dependencias no lineales entre temperatura actual, temperatura pasada y ONI. Es el más potente para horizontes cortos (1-6 meses) pero no extrapola tendencias de largo plazo. Usar como benchmark de corto plazo.

### Métricas de evaluación

Para temperatura en páramos se usan **MAE y RMSE** como métricas primarias. El MAE es preferible porque es interpretable en las mismas unidades (°C) y no penaliza desproporcionadamente los eventos extremos (que en páramos son señal real, no ruido — ADR-002). El RMSE es útil para comparar modelos cuando se quiere penalizar errores grandes.

**No usar RMSLE** para temperatura (puede tomar valores negativos) — ADR-003.

### Walk-forward validation

`walk_forward()` con `n_splits=4` simula cómo habría funcionado el modelo si se hubiera ajustado con datos históricos y se hubieran pronosticado los 6 meses siguientes, 4 veces. Esto es más realista que una simple división train/test y es el estándar para evaluación de modelos en series temporales ambientales.

In [ ]:
ts = df_clean.set_index("fecha")["temperatura"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones

Al completar el análisis con datos reales, documentar aquí:

- **Tendencia Mann-Kendall:** `trend`, `p-value`, `slope Sen (°C/mes)`, `slope Sen (°C/década)`
- **Efecto ENSO:** correlación entre `oni_lag2` y `temperatura` — signo e intensidad
- **Modelo ganador:** nombre, RMSE, MAE en walk-forward validation
- **Hallazgos ecológicos relevantes:** ¿porcentaje de meses con temperatura > 14 °C? ¿eventos de helada?
- **Recomendaciones para gestión:** ¿requiere ajuste en planes de manejo de la CAR?

### Normativa y referencias clave

| Instrumento | Aplicación en esta línea |
|---|---|
| Ley 1930 de 2018 (Ley de Páramos) | Marco general — prohíbe minería y ordena saneamiento predial |
| Resolución 886 de 2018 (MADS) | Zonificación y programas de sustitución agropecuaria |
| Resolución 963 de 2025 (IGAC) | Metodología de valoración ambiental predial en páramos |
| Guías IDEAM — Atlas de Páramos | Rangos físicos de referencia por complejo |

- Registrar decisiones metodológicas tomadas en esta sesión en `docs/decisiones.md`
- Ver ficha completa de dominio: `docs/fuentes/paramos.md`

---

## 9. Cómo adaptar a datos reales

**Paso 1 — Obtener datos:**
- Fuente principal: [IDEAM DHIME](http://dhime.ideam.gov.co/atencionciudadano/) — solicitar series históricas de temperatura para estaciones en rango altitudinal > 3.000 m.s.n.m.
- Alternativa CAR: Solicitar a la CAR regional correspondiente (CAR Cundinamarca para Sumapaz, CORPOBOYACÁ para Cocuy, etc.)
- Formato esperado: CSV con columnas de fecha y valor. Frecuencia mensual o diaria.

**Paso 2 — Preparar el archivo:**
```
data/raw/paramos.csv
```
Columnas mínimas requeridas:
```
fecha,temperatura
2015-01-31,8.4
2015-02-28,9.1
...
```
Columnas opcionales recomendadas: `precipitacion_mm`, `cobertura_pct`, `humedad_relativa_pct`, `estacion_id`.

**Paso 3 — Ajustar la celda de carga:**
Descomentar la línea:
```python
df = load_csv("data/raw/paramos.csv", date_col="fecha")
```
Y comentar el bloque de datos sintéticos.

**Paso 4 — Verificar rangos:**
Confirmar que `validate()` no arroja alertas críticas. Si temperatura > 16 °C con frecuencia, verificar altitud de la estación: puede no ser propiamente un páramo sino un bosque altoandino.

**Paso 5 — Conectar con IDEAM DHIME vía API (opcional):**
```python
from estadistica_ambiental.io.connectors import load_ideam_dhime
df = load_ideam_dhime(codigo_estacion="35010010", variable="TMAX", fecha_inicio="2010-01-01")
```